creating a composite synthetic index from all the supporting data using chow-lin algorithm

In [15]:
import pandas as pd
import numpy as np

In [16]:
# BUILD THE MONTHLY MASTER GRID
# Load Local Monthly Variables (Country-Specific)
df_exrate = pd.read_csv('exchange_rates_cleaned.csv')
df_cpi = pd.read_csv('CPI_melted.csv') 
df_trade = pd.read_csv('international_trade_in_Goods_cleaned.csv')
df_gdp = pd.read_csv('GDP_cleaned.csv')
df_bop = pd.read_csv('Balance_of_payment_cleaned.csv') 
df_external_debt = pd.read_csv('External_Debt_Final_Cleaned.csv')
print(df_exrate.head())
print(df_cpi.head())
print(df_trade.head())
print(df_gdp.head())
print(df_bop.head())
print(df_external_debt.head())

                            COUNTRY        Date  Exchange_Rate
0  Afghanistan, Islamic Republic of  2000-01-01      46.791100
1  Afghanistan, Islamic Republic of  2000-01-01      46.795800
2  Afghanistan, Islamic Republic of  2000-02-01      47.505938
3  Afghanistan, Islamic Republic of  2000-02-01      47.504800
4  Afghanistan, Islamic Republic of  2000-03-01      47.267200
                            COUNTRY  \
0  Afghanistan, Islamic Republic of   
1  Afghanistan, Islamic Republic of   
2  Afghanistan, Islamic Republic of   
3  Afghanistan, Islamic Republic of   
4  Afghanistan, Islamic Republic of   

                                  COICOP_1999                  INDEX_TYPE  \
0  Alcoholic beverages, tobacco and narcotics  Consumer price index (CPI)   
1  Alcoholic beverages, tobacco and narcotics  Consumer price index (CPI)   
2                               Communication  Consumer price index (CPI)   
3                       Clothing and footwear  Consumer price index (CPI)   
4 

In [17]:
# Load Global Monthly Variables (Macro Gravity)
df_fed = pd.read_csv('FEDFUNDS_Final_Cleaned.csv')
df_vix = pd.read_csv('VIX_Final_Cleaned.csv')
df_dxy = pd.read_csv('DXY_Final_Cleaned.csv')
df_brent = pd.read_csv('Global_price_of_Brent_Crude.csv')
df_cofer = pd.read_csv('IMF-COFER-A.W00.RAXGFXARCHF_USD_CLEANED.csv')
df_gold = pd.read_csv('Gold_rate_cleaned_data.csv')
print(df_fed.head())
print(df_vix.head())
print(df_dxy.head())
print(df_brent.head())
print(df_cofer.head())
print(df_gold.head())

   FEDFUNDS        Date
0      5.45  2000-01-01
1      5.73  2000-02-01
2      5.85  2000-03-01
3      6.02  2000-04-01
4      6.27  2000-05-01
         Date  Monthly_Avg_VIXCLS
0  2000-01-01           23.202000
1  2000-02-01           23.595500
2  2000-03-01           22.718261
3  2000-04-01           27.164211
4  2000-05-01           26.373182
         Date   DXY_Index
0  2006-01-01  100.000005
1  2006-02-01  100.211170
2  2006-03-01  100.428087
3  2006-04-01   99.743480
4  2006-05-01   97.511774
         Date  Crude_Oil_Price
0  2000-01-01        25.633333
1  2000-02-01        28.030476
2  2000-03-01        27.494348
3  2000-04-01        23.153500
4  2000-05-01        27.805217
   Date  COFER_Reserves
0  2000     4086.584364
1  2001     3849.549073
2  2002     7314.160366
3  2003     5015.670542
4  2004     4418.590733
         Date  Gold_Price
0  1833-01-01       18.93
1  1833-02-01       18.93
2  1833-03-01       18.93
3  1833-04-01       18.93
4  1833-05-01       18.93


In [18]:
df_cpi.rename(columns={'MONTH_YEAR': 'Date'}, inplace=True)

In [19]:
def clean_country_names(name):
    if not isinstance(name, str):
        return name
    # Step A: String cleansing
    name = name.strip()
    
    # Step B: Geopolitical clean expressions (stripping formal tags)
    replacements = [
        ", Islamic Republic of", ", Republic of", ", Republica Bolivariana de", 
        ", Arab Republic of", ", Arab Rep.", ", RB", ", Oriental Rep. of", 
        " Rep.", " Republic", " Islamic Rep. of", ", Plurinational State of"
    ]
    for rep in replacements:
        name = name.replace(rep, "")
        
    # Step C: Catching edge anomalies
    edge_cases = {
        "Egypt, Arab Rep.": "Egypt",
        "Venezuela, RB": "Venezuela",
        "Yemen, Rep.": "Yemen",
        "Gambia, The": "Gambia",
        "Bahamas, The": "Bahamas",
        "Congo, Dem. Rep.": "Congo",
        "Korea, Rep.": "Korea"
    }
    return edge_cases.get(name, name).strip()

# Safely unify core structural keys across data sources
datasets_to_clean = [
    df_exrate, df_cpi, df_trade, df_fed, df_vix, df_dxy, 
    df_brent, df_gold, df_external_debt, df_cofer, df_gdp, df_bop
]

for d in datasets_to_clean:
    # 1. Coordinate label matching
    for col in d.columns:
        if col.strip().upper() in ['COUNTRY', 'COUNTRY NAME', 'COUNTRY_NAME']:
            d.rename(columns={col: 'COUNTRY'}, inplace=True)
            
    # 2. Force the country string through our cleaning algorithm
    if 'COUNTRY' in d.columns:
        d['COUNTRY'] = d['COUNTRY'].apply(clean_country_names)

print("✅ Country string alignment verified!")

✅ Country string alignment verified!


In [20]:
# Create explicit key-value pairings to provide safe debugging logs
datasets_to_clean = [
    (df_exrate, 'df_exrate'), (df_cpi, 'df_cpi'), (df_trade, 'df_trade'), 
    (df_fed, 'df_fed'), (df_vix, 'df_vix'), (df_dxy, 'df_dxy'), 
    (df_brent, 'df_brent'), (df_gold, 'df_gold'), (df_external_debt, 'df_external_debt'),
    (df_cofer, 'df_cofer'), (df_gdp, 'df_gdp'), (df_bop, 'df_bop')
]
print("📦 Target registry synchronized for cleaning!")

📦 Target registry synchronized for cleaning!


In [21]:
for d, name in datasets_to_clean:
    # 1. Rename 'DATE' to 'Date' if the uppercase version exists
    if 'DATE' in d.columns: 
        d.rename(columns={'DATE': 'Date'}, inplace=True)
        
    # 2. Check if 'Date' exists before operating to prevent KeyError
    if 'Date' in d.columns:
        # 3. Use Pandas' built-in numeric type checker
        if pd.api.types.is_numeric_dtype(d['Date']):
            # Convert to string, slice the first 4 characters (turns '2000.0' into '2000'),
            # and use format='%Y' for fast, robust parsing.
            d['Date'] = pd.to_datetime(
                d['Date'].astype(str).str[:4], 
                format='%Y', 
                errors='coerce' # Safely handles NaNs/bad values
            )
        else:
            # 4. Use errors='coerce' to prevent the loop from crashing on bad strings
            d['Date'] = pd.to_datetime(d['Date'], errors='coerce')
    else:
        print(f"Warning: No 'Date' column found in dataset '{name}'. Skipping.")
        
print("All timestamps safely standardized!")


All timestamps safely standardized!


In [22]:
# 2. Crush Exchange Rates (Take the average if there are multiple rates for one month)
df_exrate = df_exrate.groupby(['COUNTRY', 'Date'], as_index=False)['Exchange_Rate'].mean()
print(df_exrate.head())
print(df_cpi.head())


       COUNTRY       Date  Exchange_Rate
0  Afghanistan 2000-01-01      46.793450
1  Afghanistan 2000-02-01      47.505369
2  Afghanistan 2000-03-01      47.267200
3  Afghanistan 2000-04-01      47.267200
4  Afghanistan 2000-05-01      47.267200
       COUNTRY                                 COICOP_1999  \
0  Afghanistan  Alcoholic beverages, tobacco and narcotics   
1  Afghanistan  Alcoholic beverages, tobacco and narcotics   
2  Afghanistan                               Communication   
3  Afghanistan                       Clothing and footwear   
4  Afghanistan                               Communication   

                   INDEX_TYPE       Date  CPI_VALUE  
0  Consumer price index (CPI) 2000-01-01  61.138141  
1  Consumer price index (CPI) 2000-01-01  61.138141  
2  Consumer price index (CPI) 2000-01-01  61.138141  
3  Consumer price index (CPI) 2000-01-01  61.138141  
4  Consumer price index (CPI) 2000-01-01  61.138141  


In [23]:
# 3. Crush CPI (Take the average across all product categories to get the National CPI)
df_cpi = df_cpi.groupby(['COUNTRY', 'Date'], as_index=False)['CPI_VALUE'].mean()


In [24]:
# 4. Flatten Trade Data (Pivot Exports/Imports into their own separate X-variables)
if 'INDICATOR' in df_trade.columns:
    df_trade = df_trade.pivot_table(
        index=['COUNTRY', 'Date'], 
        columns='INDICATOR', 
        values='Trade_in_Goods',
        aggfunc='mean'
    ).reset_index()
    df_trade.columns.name = None # Remove pivot name
print(df_trade.head())

   COUNTRY       Date  Export price index (EPI)  Exports of goods  \
0  Albania 2000-01-01                       NaN          8.503182   
1  Albania 2000-02-01                       NaN          8.503182   
2  Albania 2000-03-01                       NaN          8.503182   
3  Albania 2000-04-01                       NaN          8.503182   
4  Albania 2000-05-01                       NaN          8.503182   

   Exports of goods, Price deflator  Exports of goods, Volume index  \
0                               NaN                             NaN   
1                               NaN                             NaN   
2                               NaN                             NaN   
3                               NaN                             NaN   
4                               NaN                             NaN   

   Import price index  Imports of goods  Imports of goods, Price deflator  \
0                 NaN         11.517991                               NaN   
1   

In [25]:
# MASTER MERGE - taking the exchange rate dataframe as the base and merging in the rest of the variables
# =================================================================
# PHASE 2: DEDUPLICATED MASTER UNIFICATION GRID
# =================================================================
import pandas as pd
import numpy as np

print("⚙️ Deduplicating records and building a strict 1-to-1 Time Grid...")

# Deduplicate every country-month grouping by taking its mathematical mean
df_exrate_clean = df_exrate.groupby(['COUNTRY', 'Date'], as_index=False)['Exchange_Rate'].mean()
df_cpi_clean = df_cpi.groupby(['COUNTRY', 'Date'], as_index=False)['CPI_VALUE'].mean()

# Pivot or clean the trade parameters strictly to unique combinations
if 'INDICATOR' in df_trade.columns:
    df_trade_pivot = df_trade.pivot_table(index=['COUNTRY', 'Date'], columns='INDICATOR', values='Trade_in_Goods').reset_index()
    df_trade_pivot.rename(columns={'Exports of goods': 'Exports_of_goods', 'Imports of goods': 'Imports_of_goods'}, inplace=True)
else:
    df_trade_pivot = df_trade.groupby(['COUNTRY', 'Date'], as_index=False).mean()

df_debt_clean = df_external_debt.groupby(['COUNTRY', 'Date'], as_index=False)['External_Debt'].mean()
df_gdp_clean = df_gdp.groupby(['COUNTRY', 'Date'], as_index=False)['GDP'].mean()
df_bop_clean = df_bop.groupby(['COUNTRY', 'Date'], as_index=False)['BOP_USD'].mean()

# Re-run Master Assembly on perfectly clean components
df_monthly = df_exrate_clean.copy()
df_monthly = pd.merge(df_monthly, df_cpi_clean, on=['COUNTRY', 'Date'], how='left')
df_monthly = pd.merge(df_monthly, df_trade_pivot, on=['COUNTRY', 'Date'], how='left')
df_monthly = pd.merge(df_monthly, df_debt_clean, on=['COUNTRY', 'Date'], how='left')
df_monthly = pd.merge(df_monthly, df_gdp_clean, on=['COUNTRY', 'Date'], how='left')
df_monthly = pd.merge(df_monthly, df_bop_clean, on=['COUNTRY', 'Date'], how='left')

# Broadcast uniform macro environment benchmarks to all rows
df_monthly = pd.merge(df_monthly, df_fed.groupby('Date')['FEDFUNDS'].mean().reset_index(), on='Date', how='left')
df_monthly = pd.merge(df_monthly, df_vix.groupby('Date')['Monthly_Avg_VIXCLS'].mean().reset_index(), on='Date', how='left')
df_monthly = pd.merge(df_monthly, df_dxy.groupby('Date')['DXY_Index'].mean().reset_index(), on='Date', how='left')
df_monthly = pd.merge(df_monthly, df_brent.groupby('Date')['Crude_Oil_Price'].mean().reset_index(), on='Date', how='left')
df_monthly = pd.merge(df_monthly, df_gold.groupby('Date')['Gold_Price'].mean().reset_index(), on='Date', how='left')
df_monthly = pd.merge(df_monthly, df_cofer.groupby('Date')['COFER_Reserves'].mean().reset_index(), on='Date', how='left')

# Chronological sorting for structural stability
df_monthly = df_monthly.sort_values(by=['COUNTRY', 'Date']).reset_index(drop=True)

# Propagate annual variables safely
annual_columns = ['GDP', 'BOP_USD', 'External_Debt']
df_monthly[annual_columns] = df_monthly.groupby('COUNTRY')[annual_columns].ffill()

print(f"🏁 Clean Master DataFrame Integrated! Controlled Shape: {df_monthly.shape}")
# Your row count should now land closely around 30,000 to 35,000 rows max! 

⚙️ Deduplicating records and building a strict 1-to-1 Time Grid...
🏁 Clean Master DataFrame Integrated! Controlled Shape: (63945, 21)


In [26]:
# Sort by timeline to execute proper chronological propagation
df_monthly = df_monthly.sort_values(by=['COUNTRY', 'Date']).reset_index(drop=True)

# List of columns that originate from annual reports and need to be broadcasted monthly
annual_columns = ['GDP', 'Percent of GDP', 'BOP_USD', 'COFER_Reserves']
available_annuals = [col for col in annual_columns if col in df_monthly.columns]

# Forward fill within each country group to propagate annual constants down all 12 months
df_monthly[available_annuals] = df_monthly.groupby('COUNTRY')[available_annuals].ffill()

print("✅ Annual Macro metrics successfully broadcasted across monthly steps!")
display(df_monthly[['COUNTRY', 'Date', 'GDP', 'BOP_USD', 'COFER_Reserves']].dropna().head(12))

✅ Annual Macro metrics successfully broadcasted across monthly steps!


,COUNTRY,Date,GDP,BOP_USD,COFER_Reserves
96,Afghanistan,2008-01-01,5.803678e+10,609.194411,5799.366269
97,Afghanistan,2008-02-01,5.803678e+10,609.194411,5799.366269
98,Afghanistan,2008-03-01,5.803678e+10,609.194411,5799.366269
99,Afghanistan,2008-04-01,5.803678e+10,609.194411,5799.366269
100,Afghanistan,2008-05-01,5.803678e+10,609.194411,5799.366269
101,Afghanistan,2008-06-01,5.803678e+10,609.194411,5799.366269
102,Afghanistan,2008-07-01,5.803678e+10,609.194411,5799.366269
103,Afghanistan,2008-08-01,5.803678e+10,609.194411,5799.366269
104,Afghanistan,2008-09-01,5.803678e+10,609.194411,5799.366269
105,Afghanistan,2008-10-01,5.803678e+10,609.194411,5799.366269


In [27]:
print("Final Shape:", df_monthly.shape)
display(df_monthly.head(20))
df_monthly.to_csv('composite_pool.csv', index=False)

Final Shape: (63945, 21)


,COUNTRY,Date,Exchange_Rate,CPI_VALUE,Export price index (EPI),Exports of goods,"Exports of goods, Price deflator","Exports of goods, Volume index",Import price index,Imports of goods,...,"Imports of goods, Volume index",External_Debt,GDP,BOP_USD,FEDFUNDS,Monthly_Avg_VIXCLS,DXY_Index,Crude_Oil_Price,Gold_Price,COFER_Reserves
0,Afghanistan,2000-01-01,46.793450,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.256738e+10,NaN,5.45,23.202000,NaN,25.633333,284.32,4086.584364
1,Afghanistan,2000-02-01,47.505369,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.256738e+10,NaN,5.73,23.595500,NaN,28.030476,299.86,4086.584364
2,Afghanistan,2000-03-01,47.267200,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.256738e+10,NaN,5.85,22.718261,NaN,27.494348,286.39,4086.584364
3,Afghanistan,2000-04-01,47.267200,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.256738e+10,NaN,6.02,27.164211,NaN,23.153500,279.69,4086.584364
4,Afghanistan,2000-05-01,47.267200,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.256738e+10,NaN,6.27,26.373182,NaN,27.805217,275.19,4086.584364
5,Afghanistan,2000-06-01,47.267200,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.256738e+10,NaN,6.53,21.540000,NaN,30.483182,285.73,4086.584364
6,Afghanistan,2000-07-01,47.451148,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.256738e+10,NaN,6.54,19.893000,NaN,28.894286,281.59,4086.584364
7,Afghanistan,2000-08-01,47.504800,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.256738e+10,NaN,6.50,18.088696,NaN,31.629130,274.47,4086.584364
8,Afghanistan,2000-09-01,47.504800,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.256738e+10,NaN,6.52,19.687500,NaN,33.360476,273.68,4086.584364
9,Afghanistan,2000-10-01,47.504800,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.256738e+10,NaN,6.51,25.200000,NaN,31.302727,270.00,4086.584364


STEP 2 & 3: PCA SYNTHETIC INDEX & CHOW-LIN TRIANGULATION

In [28]:
import pip
!pip install tempdisagg

In [29]:
import os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [30]:
#Inject R's core pathways into the Windows Environment
R_HOME = r"C:\Program Files\R\R-4.6.0"
os.environ['R_HOME'] = R_HOME
# Add the 'bin\x64' folder to the system PATH so stats.dll can find its dependencies
os.environ['PATH'] = R_HOME + r"\bin\x64;" + os.environ.get('PATH', '')

In [31]:
# NOW it is safe to import the rpy2 bridge
import rpy2.robjects as ro
from rpy2.robjects import FloatVector
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

In [32]:
# Verify the package loads without crashing
try:
    tempdisagg_r = importr('tempdisagg')
    print("R Environment and tempdisagg loaded successfully!")
except Exception as e:
    print("Error: R package 'tempdisagg' still not found.")
    raise e

R Environment and tempdisagg loaded successfully!


In [33]:
df_nfa = pd.read_csv('NFA_Annual_Anchors_Cleaned.csv')
print(df_nfa.head())


       COUNTRY        Date  Assets, Claims on Central Government (CBS)  \
0  Afghanistan  2006-12-31                                 2003.882126   
1  Afghanistan  2007-12-31                                16706.636211   
2  Afghanistan  2008-12-31                                19227.020024   
3  Afghanistan  2009-12-31                                13650.763048   
4  Afghanistan  2010-12-31                                13650.763048   

   Assets, Claims on Nonresidents (CBS)  \
0                         100285.840640   
1                         133299.139871   
2                         161734.421728   
3                         205060.860621   
4                         226880.956341   

   Assets, Claims on Other depository corporations (CBS)  \
0                                               0.00       
1                                               0.00       
2                                               0.00       
3                                               0.00    

In [34]:
# =================================================================
# THE ENGINE: PCA SYNTHETIC INDEX & OFFICIAL R CHOW-LIN (via rpy2)
# =================================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import rpy2.robjects as ro
from rpy2.robjects import FloatVector
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri

# -----------------------------------------------------------------
# Remind Pandas that 'Date' is a mathematical Time object
# -----------------------------------------------------------------
df_nfa['Date'] = pd.to_datetime(df_nfa['Date'])
df_monthly['Date'] = pd.to_datetime(df_monthly['Date'])

def create_index_and_disaggregate(country, df_annual, df_monthly):
    # 1. Extract Annual NFA Anchor
    nfa_col = 'Net (assets minus liabilities), Net foreign assets (CBS)'
    
    nfa_a = df_annual[df_annual['COUNTRY'] == country].set_index('Date')[nfa_col].dropna()
    
    # 2. Extract Features for the Synthetic Index (NO EXCHANGE RATES)
    features = ['Monthly_Avg_VIXCLS', 'FEDFUNDS', 'CPI_VALUE', 'Gold_Price', 'Crude_Oil_Price', ] 
    country_monthly = df_monthly[df_monthly['COUNTRY'] == country].set_index('Date').dropna(subset=features)
    
    if len(nfa_a) < 3 or len(country_monthly) < 24:
        return None 
        
    # 3. USE THE FULL TIMELINE
    nfa_train = nfa_a
    monthly_train = country_monthly
    
    if len(nfa_train) < 3:
        return None

    try:
        # 4. ALIGN THE MATHEMATICAL GRID 
        start_year = max(nfa_train.index.min().year, monthly_train.index.min().year)
        end_year = min(nfa_train.index.max().year, monthly_train.index.max().year)
        
        nfa_train = nfa_train[(nfa_train.index.year >= start_year) & (nfa_train.index.year <= end_year)]
        perfect_months = pd.date_range(start=f"{start_year}-01-01", end=f"{end_year}-12-01", freq='MS')
        monthly_train = monthly_train.reindex(perfect_months).ffill().bfill()

        # 5. CREATE THE SYNTHETIC MACRO INDEX (PCA)
        scaler = StandardScaler()
        scaled_features = scaler.fit_transform(monthly_train[features])
        pca = PCA(n_components=1)
        synthetic_index = pca.fit_transform(scaled_features).flatten()
        
        # 6. THE R-BRIDGE: SEND DATA TO R FOR CHOW-LIN
        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv['Y_py'] = FloatVector(nfa_train.values)
            ro.globalenv['X_py'] = FloatVector(synthetic_index)
            ro.globalenv['start_yr'] = start_year
            
            # THE FIX: Explicitly forcing R to load the libraries inside the bridge
            r_script = """
                library(stats)
                library(tempdisagg)
                Y_ts <- ts(Y_py, start=c(start_yr, 1), frequency=1)
                X_ts <- ts(X_py, start=c(start_yr, 1), frequency=12)
                model <- td(Y_ts ~ X_ts, conversion="average", method="chow-lin-maxlog")
                as.numeric(predict(model))
            """
            nfa_monthly_train = ro.r(r_script)
        
        # 7. Format the R output back into a Pandas DataFrame
        temp_df = pd.DataFrame({
            'Date': perfect_months, 
            'NFA_Triangulated': np.array(nfa_monthly_train),
            'COUNTRY': country
        })
        return temp_df
    
    except Exception as e:
        # Silently skip countries where the math fails (e.g., singular matrices)
        return None

# =================================================================
# EXECUTE THE LOOP ACROSS ALL COUNTRIES
# =================================================================
print("🧠 Building Synthetic Indices & Routing to R for Chow-Lin... (This may take 30-60 seconds)")
triangulated_data = []
countries = df_monthly['COUNTRY'].unique()

for country in countries:
    df_tri = create_index_and_disaggregate(country, df_nfa, df_monthly) 
    if df_tri is not None:
        triangulated_data.append(df_tri)

# Combine everything into one final DataFrame
df_nfa_all = pd.concat(triangulated_data, ignore_index=True)

print("✅ PCA Synthetic Index & Chow-Lin Triangulation Complete!")
display(df_nfa_all.head(10))

🧠 Building Synthetic Indices & Routing to R for Chow-Lin... (This may take 30-60 seconds)


R callback write-console: High frequency series shorter than low frequency. Discarding low frequency after 2023.
  


✅ PCA Synthetic Index & Chow-Lin Triangulation Complete!


,Date,NFA_Triangulated,COUNTRY
0,2006-01-01,93218.758449,Afghanistan
1,2006-02-01,91517.798182,Afghanistan
2,2006-03-01,90962.011546,Afghanistan
3,2006-04-01,93750.179423,Afghanistan
4,2006-05-01,95165.963962,Afghanistan
5,2006-06-01,94364.245018,Afghanistan
6,2006-07-01,96237.238874,Afghanistan
7,2006-08-01,98560.803681,Afghanistan
8,2006-09-01,98252.251557,Afghanistan
9,2006-10-01,101217.619289,Afghanistan


In [35]:
df_nfa_all.to_csv('NFA_Monthly_Triangulated.csv', index=False)